In [ ]:
import pandas as pd
import nltk

from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from nltk.stem import PorterStemmer

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import accuracy_score

# 1. Load Dataset
df = pd.read_csv("combined_data.csv")

# Convert labels if needed
if df['label'].dtype == object:
    df['label'] = df['label'].map({'spam': 1, 'ham': 0})

# 2. NLTK Preprocessing Function
stop_words = set(stopwords.words('english'))
stemmer = PorterStemmer()

def nltk_tokenizer(text):
    tokens = word_tokenize(str(text).lower())
    tokens = [
        stemmer.stem(word)
        for word in tokens
        if word.isalpha() and word not in stop_words
    ]
    return tokens

# 3. Train-Test Split (RAW text)
X_train, X_test, y_train, y_test = train_test_split(
    df['text'],
    df['label'],
    test_size=0.2,
    random_state=42
)

# 4. TF-IDF using NLTK tokenizer
vectorizer = TfidfVectorizer(
    tokenizer=nltk_tokenizer,  
    lowercase=True,
    min_df=1,                  
    max_df=0.95,
    ngram_range=(1,1)
)

X_train_tfidf = vectorizer.fit_transform(X_train)
X_test_tfidf = vectorizer.transform(X_test)

# 5. Train Multinomial Naive Bayes
model = MultinomialNB()
model.fit(X_train_tfidf, y_train)

# 6. Evaluation
y_pred = model.predict(X_test_tfidf)
print("Accuracy:", accuracy_score(y_test, y_pred))

# 7. Predict on New Email
def predict_email(text):
    vec = vectorizer.transform([text])
    print("Non-zero features:", vec.nnz)
    pred = model.predict(vec)[0]
    return "Spam" if pred == 1 else "Ham"

# Example tests
print(predict_email("Hello, how are you doing today?"))
print(predict_email("Congratulations! You won a free prize. Click now!"))


c:\Users\Shaarukesh\anaconda3\Lib\site-packages\sklearn\feature_extraction\text.py:517: UserWarning: The parameter 'token_pattern' will not be used since 'tokenizer' is not None'
  warnings.warn(


Accuracy: 0.9766327142001199
Non-zero features: 2
Spam
Non-zero features: 4
Spam


In [3]:
print(predict_email("Invitation from an unknown sender: Graduate Interns Onboarding @ Mon 2 Feb 2026 7pm - 8pm"))
print(predict_email("Limited time offer! Get 50% off on all products. Hurry, shop now!"))

Non-zero features: 8
Ham
Non-zero features: 7
Spam
